# Child Safety Monitor — Benchmark Tool (Annotated)

## What this notebook is

This is a **step-by-step walkthrough** of `benchmark.py` — the Phase 1 benchmark tool for the Child Safety Monitor. Every code block is paired with an explanation of **what it does** and **why it is designed that way**.

> ⚠️ **This notebook is for learning only.**  
> `cv2.imshow()` crashes the Jupyter kernel on macOS. To actually run the benchmark, use the terminal:
> ```bash
> cd /Users/shirch/Desktop/Objection\ Detection
> python benchmark.py            # full benchmark
> python benchmark.py --report   # view report from saved results
> python benchmark.py --fp       # false-positive test only
> ```

---

## Why do this benchmark at all?

Phase 1 uses YOLOWorld (zero-shot) to detect custom dangerous items like lighters, pill bottles, and button batteries. Zero-shot means no training data was used — the model guesses based on the text description alone. Accuracy varies widely by class.

Before spending weeks collecting training images in Phase 2, we need to know **which classes actually need training** and which are already good enough. The benchmark answers:

- **Detection rate** — out of all frames where the object is visible, what fraction did the model actually detect it? (Target: ≥ 70%)
- **False positive rate** — with no dangerous objects present, how often does the model incorrectly fire an alert?

Classes scoring below 70% DR → priority targets for Phase 2 data collection.  
Classes scoring ≥ 70% DR → skip training for now, focus resources elsewhere.

---

## Test matrix

```
21 classes × 3 distances = 63 detection-rate tests
                         + 1 false-positive test (60 s, no objects)
                         ─────────────────────────────────────────
                           ~35 minutes total if run without skipping
```

The 3 distances match Phase 2 image collection requirements:
- **Near** (0.5–1 m) — object is large in frame
- **Mid**  (1.5–3 m) — object is medium-sized
- **Far**  (4–6 m)   — object is small; most challenging for detection

---

## Controls during a test

| Key | Action |
|---|---|
| `SPACE` | Start recording the current 30-second test |
| `S` | Skip this test (if you don't have the object) |
| `Q` / `ESC` | Quit and save progress — resumes from here next run |

---
## Step 1 — SSL Fix and Imports

Same SSL patch as `yoloworld_demo.py`: Python 3.13 on macOS doesn't use the system certificate store, so model downloads fail without this line.

New imports compared to the main demo:

| Library | Role |
|---|---|
| `json` | Save and load benchmark results incrementally to `benchmark_results.json` |
| `argparse` | Parse command-line flags (`--report`, `--fp`) |
| `pathlib.Path` | Clean file path handling for the results and report files |
| `datetime` | Timestamp each test result so you know when it was recorded |

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import cv2
import sys
import json
import time
import platform
import argparse
import numpy as np
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO, YOLOWorld

---
## Step 2 — Configuration

Most constants are shared with `yoloworld_demo.py` (same models, same thresholds, same device). Two new ones specific to benchmarking:

| Constant | Value | Reasoning |
|---|---|---|
| `RECORD_SECONDS = 30` | 30 s | Long enough to get ~900 frames (at 30 FPS) for a statistically meaningful detection rate. Short enough that the full benchmark stays manageable. |
| `FP_TEST_SECONDS = 60` | 60 s | Twice as long, because false positives are rare events — a 30-second window might not capture any even if the model is problematic. 60 s gives a more reliable rate-per-minute figure. |
| `RESULTS_FILE` | `.json` | JSON is human-readable, easy to inspect, and simple to update incrementally. |
| `REPORT_FILE` | `.md` | Markdown renders nicely in VS Code, GitHub, and Jupyter — easy to share. |

In [ ]:
CAMERA_INDEX  = 0
CAMERA_WIDTH  = 1280
CAMERA_HEIGHT = 720
DEVICE        = "mps"
IMGSZ         = 640

SEG_CONF   = 0.25
WORLD_CONF = 0.20
IOU        = 0.45

RECORD_SECONDS  = 30    # seconds to record per class × distance test
FP_TEST_SECONDS = 60    # seconds for the false-positive test
WORLD_SKIP      = 3     # run YOLOWorld every N frames (same as main demo)

RESULTS_FILE = Path("benchmark_results.json")
REPORT_FILE  = Path("benchmark_report.md")

---
## Step 3 — Class and Distance Definitions

**Why benchmark all 3 distances separately, not just one?**

Detection rate at close range and at far range are completely different problems. A model might detect a lighter perfectly at 0.5 m but completely miss it at 4 m — not because it doesn't know what a lighter is, but because the object is too small in the frame at far distance.

If we only test at near distance and see 90%, we might incorrectly conclude the class is fine. In a real home monitoring scenario, the camera is often far from where a child plays. Far-distance performance is the most important and most likely to fail.

The distance breakdown also directly informs Phase 2 image collection: if far DR is low, you know to collect more far-distance training images for that class.

In [ ]:
COCO_CLASSES: set[str] = {
    "knife", "scissors", "bottle", "wine glass",
    "fork", "microwave", "oven", "toaster",
}

CUSTOM_CLASSES: list[str] = [
    "lighter",
    "matches",
    "pill bottle",
    "medicine bottle",
    "button battery",
    "power cord",
    "extension cord",
    "plastic bag",
    "candle",
    "iron",
    "needle",
    "syringe",
    "cleaning spray bottle",
]

# Sorted COCO first, then custom — determines the order tests are run
ALL_DANGER_CLASSES = sorted(COCO_CLASSES) + CUSTOM_CLASSES

DISTANCES = ["near", "mid", "far"]
DISTANCE_LABELS = {
    "near": "NEAR  (0.5 – 1 m)",
    "mid":  "MID   (1.5 – 3 m)",
    "far":  "FAR   (4 – 6 m)  ",
}

total_tests = len(ALL_DANGER_CLASSES) * len(DISTANCES)
print(f"Total test slots: {len(ALL_DANGER_CLASSES)} classes × {len(DISTANCES)} distances = {total_tests}")
print(f"Estimated time (no skips): {total_tests * RECORD_SECONDS // 60} min detection + {FP_TEST_SECONDS // 60} min FP test")

---
## Step 4 — State Machine Constants

Each test has three UI states:

```
IDLE  ──[press SPACE]──►  RECORDING  ──[30s elapsed]──►  DONE  ──[2s or any key]──► next test
         [press S]                                         │
            └─────────────────────────────────────────────►  skip to next test
```

**Why a state machine instead of just `time.sleep(30)`?**  
A sleep-based approach would freeze the video feed during recording — you'd see a static frame and have no idea whether the model is detecting anything. The state machine keeps the camera loop running every frame, updates the UI with live detection results and a countdown timer, and simply tracks which phase of the test we're in.

In [ ]:
IDLE      = "idle"       # waiting for SPACE — object placement time
RECORDING = "recording"  # actively counting detected frames
DONE      = "done"       # showing result summary before auto-advancing

---
## Step 5 — Results Storage Functions

**Why save incrementally after every single test?**

The full benchmark takes ~35 minutes. If the script crashes or you accidentally close the window halfway through, all progress would be lost with a write-at-the-end approach. By writing to `benchmark_results.json` after each individual test, the worst case is losing only the test that was currently running.

**Results file structure:**
```json
{
  "knife": {
    "near": { "total_frames": 897, "detected_frames": 854, "detection_rate": 0.9520, "timestamp": "2026-04-27T15:30:00" },
    "mid":  { ... },
    "far":  { ... }
  },
  "lighter": { ... },
  "_false_positive_test": {
    "total_frames": 1800,
    "fp_per_class": { "knife": 3, "lighter": 12, ... },
    "fp_rate_per_min": { "knife": 0.1, "lighter": 0.4, ... }
  }
}
```

**`is_done()`** checks whether a class × distance slot already has a result. This is what allows the benchmark to resume from where it stopped — already-completed slots are skipped automatically.

In [ ]:
def load_results() -> dict:
    """Load existing results from disk, or return empty dict if no file yet."""
    if RESULTS_FILE.exists():
        with open(RESULTS_FILE) as f:
            return json.load(f)
    return {}


def save_results(results: dict) -> None:
    """Write the full results dict to disk."""
    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)


def is_done(results: dict, class_name: str, distance: str) -> bool:
    """True if this class × distance slot already has a recorded result."""
    return class_name in results and distance in results.get(class_name, {})


def record_result(results: dict, class_name: str, distance: str,
                  total: int, detected: int) -> None:
    """Compute detection rate and save this test result immediately."""
    if class_name not in results:
        results[class_name] = {}
    dr = round(detected / total, 4) if total > 0 else 0.0
    results[class_name][distance] = {
        "total_frames":    total,
        "detected_frames": detected,
        "detection_rate":  dr,          # 0.0 – 1.0
        "timestamp":       datetime.now().isoformat(timespec="seconds"),
    }
    save_results(results)   # write to disk immediately


def record_fp_result(results: dict, fp_counts: dict[str, int],
                     total_frames: int, duration_s: float) -> None:
    """Save false-positive test results. fp_rate_per_min normalises for easy comparison."""
    results["_false_positive_test"] = {
        "total_frames":     total_frames,
        "duration_seconds": round(duration_s, 1),
        "fp_per_class":     fp_counts,
        # Normalise to per-minute so it's comparable regardless of test duration
        "fp_rate_per_min": {
            k: round(v / duration_s * 60, 2)
            for k, v in fp_counts.items()
        },
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    save_results(results)

---
## Step 6 — Overlay Drawing Functions

Three visual elements are drawn on each frame during a test:

**`draw_status_bar()`** — a dark bar at the top of the frame showing the current test name, distance, and live detection count. The text colour changes to green when the target is detected this frame, staying blue/red when it isn't. This gives immediate visual feedback so you can see exactly when the model picks up the object.

**`draw_progress_bar()`** — a thin bar at the very bottom of the frame. During recording it fills left-to-right over 30 seconds so you always know how much time is left without reading numbers. Colour: red during recording, green when done.

**`draw_detection_highlight()`** — draws a bounding box around each detected object. Green = the target class we're currently benchmarking. Grey = something else was detected (e.g., a person in the background). This lets you see which objects the model is responding to.

In [ ]:
STATUS_H = 80   # height in pixels of the top status bar

def draw_status_bar(frame: np.ndarray, lines: list[tuple[str, tuple]]) -> None:
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (0, 0), (w, STATUS_H), (20, 20, 20), -1)   # dark background
    for i, (text, colour) in enumerate(lines):
        y = 22 + i * 26
        cv2.putText(frame, text, (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, colour, 2, cv2.LINE_AA)


def draw_progress_bar(frame: np.ndarray, progress: float,
                      colour: tuple = (0, 200, 80)) -> None:
    h, w = frame.shape[:2]
    bar_h  = 12
    filled = int(w * min(progress, 1.0))   # clamp to avoid drawing beyond frame edge
    cv2.rectangle(frame, (0, h - bar_h), (w, h), (50, 50, 50), -1)        # grey background
    cv2.rectangle(frame, (0, h - bar_h), (filled, h), colour, -1)          # coloured fill


def draw_detection_highlight(frame: np.ndarray, box: tuple,
                              label: str, detected: bool) -> None:
    x1, y1, x2, y2 = box
    colour = (0, 255, 0) if detected else (100, 100, 100)   # green = target, grey = other
    cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
    cv2.putText(frame, label, (x1, max(y1 - 6, 14)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, colour, 2, cv2.LINE_AA)

---
## Step 7 — Core Detection Function

**What:** Runs both models on a single frame and returns everything detected.

**Why is this extracted into its own function?**  
Both the detection-rate test and the false-positive test need to run inference. Putting the inference logic here avoids duplicating it. Both test functions just call `detect_classes_in_frame()` and interpret the results differently.

**Key difference from `yoloworld_demo.py`:** Here we use `.predict()` instead of `.track()`. During benchmarking we don't care about consistent person IDs across frames — we only want to know whether a specific class is detected. `.predict()` is slightly faster because it skips the tracking association step.

**Return values:**
- `detected_set` — the set of all class names found this frame (used for fast membership checks: `"knife" in detected_set`)
- `seg_boxes` — list of `(class_name, xyxy)` from the seg model (for drawing)
- `world_boxes` — list of `(class_name, xyxy)` from YOLOWorld cache (for drawing)

In [ ]:
def detect_classes_in_frame(frame, seg_model, world_model,
                             frame_count: int,
                             cached_world: list) -> tuple[set[str], list, list]:
    detected: set[str]             = set()
    seg_boxes:   list[tuple]       = []
    world_boxes: list[tuple]       = []

    # Primary model: runs every frame
    seg_res = seg_model.predict(
        source=frame, conf=SEG_CONF, iou=IOU,
        device=DEVICE, imgsz=IMGSZ, verbose=False,
    )
    if seg_res and seg_res[0].boxes is not None:
        for box in seg_res[0].boxes:
            cls_name = seg_model.names[int(box.cls[0].item())]
            xyxy = tuple(map(int, box.xyxy[0].tolist()))
            seg_boxes.append((cls_name, xyxy))
            detected.add(cls_name)

    # Secondary model: runs every WORLD_SKIP frames, results cached between runs
    if frame_count % WORLD_SKIP == 0:
        w_res = world_model.predict(
            source=frame, conf=WORLD_CONF, iou=IOU,
            device=DEVICE, imgsz=IMGSZ, verbose=False,
        )
        cached_world.clear()
        if w_res and w_res[0].boxes is not None:
            for box in w_res[0].boxes:
                cls_name = world_model.names[int(box.cls[0].item())]
                xyxy = tuple(map(int, box.xyxy[0].tolist()))
                cached_world.append((cls_name, xyxy))

    # Always include cached world detections in this frame's results
    for cls_name, xyxy in cached_world:
        world_boxes.append((cls_name, xyxy))
        detected.add(cls_name)

    return detected, seg_boxes, world_boxes

---
## Step 8 — Single Detection-Rate Test

**What:** Runs one class × distance combination. You place the object, press SPACE, and the script records for 30 seconds.

**How detection rate is calculated:**
```
detection_rate = detected_frames / total_frames
```
Every frame during the 30-second recording window is counted as `total_frames`. If the target class appears in `detected_set` for that frame, `detected_frames` is incremented. The final ratio gives a detection rate between 0 and 1.

**Why count frames instead of time?**  
Frame-based counting is more precise. If the GPU slows down and FPS drops to 15 for a moment, a time-based counter would still advance but we'd have fewer actual detection samples. Frame counting always reflects the true number of inference results.

**Return values:**
- `(total_frames, detected_frames)` — test completed normally
- `None` — user pressed Q (quit signal to stop all tests)
- `(-1, -1)` — user pressed S (skip this test, move to next)

In [ ]:
def run_single_test(cap, seg_model, world_model,
                    class_name: str, distance: str,
                    test_num: int, total_tests: int) -> tuple[int, int] | None:

    window = "Benchmark"
    state  = IDLE

    total_frames    = 0
    detected_frames = 0
    record_start    = 0.0
    cached_world:   list = []
    frame_count     = 0

    dist_label   = DISTANCE_LABELS[distance]
    source_label = "COCO model" if class_name in COCO_CLASSES else "YOLOWorld"

    # Print instructions to terminal so user knows what to do even without the window
    print(f"\n{'─'*55}")
    print(f"  Test {test_num}/{total_tests}")
    print(f"  Class   : {class_name.upper()}")
    print(f"  Distance: {dist_label}")
    print(f"  Model   : {source_label}")
    print(f"  → Place the object at {dist_label.strip()} and press SPACE")
    print(f"{'─'*55}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        now = time.time()

        detected_set, seg_boxes, world_boxes = detect_classes_in_frame(
            frame, seg_model, world_model, frame_count, cached_world
        )

        target_found = class_name in detected_set
        annotated    = frame.copy()

        # Draw all boxes; target class gets green, everything else grey
        for (cn, box) in seg_boxes + world_boxes:
            draw_detection_highlight(annotated, box, cn, cn == class_name)

        # ── State: IDLE (waiting for SPACE) ──────────────────────────────────
        if state == IDLE:
            lines = [
                (f"[{test_num}/{total_tests}] Testing: {class_name.upper()}  |  {dist_label}",
                 (255, 255, 255)),
                ("SPACE = start recording  |  S = skip  |  Q = quit",
                 (160, 160, 160)),
            ]
            draw_status_bar(annotated, lines)
            draw_progress_bar(annotated, 0.0, (180, 180, 0))

        # ── State: RECORDING (counting frames) ────────────────────────────────
        elif state == RECORDING:
            elapsed   = now - record_start
            remaining = max(0, RECORD_SECONDS - elapsed)

            # Count this frame
            total_frames += 1
            if target_found:
                detected_frames += 1

            dr_so_far = detected_frames / total_frames if total_frames else 0

            # Text turns green when model is detecting the target this frame
            text_colour = (80, 220, 80) if target_found else (80, 80, 255)
            lines = [
                (f"RECORDING  {class_name.upper()}  |  {dist_label}", text_colour),
                (f"{remaining:.1f}s left  |  Detected: {detected_frames}/{total_frames}"
                 f"  ({dr_so_far*100:.1f}%)",
                 (255, 255, 255)),
            ]
            draw_status_bar(annotated, lines)
            draw_progress_bar(annotated, elapsed / RECORD_SECONDS, (0, 0, 200))

            if elapsed >= RECORD_SECONDS:
                state = DONE   # 30 seconds elapsed → move to result display

        # ── State: DONE (showing result for 2 seconds before auto-advance) ───
        elif state == DONE:
            dr         = detected_frames / total_frames if total_frames else 0
            status_str = "✓ PASS (>=70%)" if dr >= 0.70 else "✗ FAIL (<70%) -> needs training"
            colour_st  = (80, 220, 80) if dr >= 0.70 else (80, 80, 255)
            lines = [
                (f"DONE  {class_name.upper()}  |  {dist_label}", (255, 255, 255)),
                (f"Detection rate: {dr*100:.1f}%  |  {status_str}", colour_st),
            ]
            draw_status_bar(annotated, lines)
            draw_progress_bar(annotated, 1.0, (0, 200, 80))

        cv2.imshow(window, annotated)

        # ── Key handling ──────────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF

        if key in [ord("q"), ord("Q"), 27]:     # quit: save and stop all tests
            return None
        elif key == ord(" ") and state == IDLE: # space: begin recording
            state        = RECORDING
            record_start = time.time()
            total_frames    = 0
            detected_frames = 0
            print("  ► Recording…")
        elif key in [ord("s"), ord("S")]:        # skip this test
            print("  Skipped.")
            return (-1, -1)
        elif state == DONE:
            # Auto-advance after 2 seconds, or immediately on any key press
            if not hasattr(run_single_test, "_done_time"):
                run_single_test._done_time = now
            elif now - run_single_test._done_time > 2.0 or key != 255:
                del run_single_test._done_time
                return (total_frames, detected_frames)

    return None

---
## Step 9 — False Positive Test

**What:** Records 60 seconds with **no dangerous objects in the camera view** and counts how often each class is incorrectly detected.

**Why this matters:**  
A model that fires alerts constantly — even when there's nothing dangerous present — is useless in practice. Parents would turn it off within minutes. A high false positive rate means the model is confusing ordinary household objects (a remote control, a pen, a bottle of water) for dangerous ones.

**What you'll see during this test:**  
If a box appears on screen labeled `FP: knife`, that means the model thought it saw a knife when there was none. These false detections are highlighted in green (the same as a target hit) to make them very visible.

**Threshold:** A false positive rate above **2 per minute** for any class is a concern — that would mean an alert fires roughly every 30 seconds, which would cause serious alert fatigue.

**Return value:** `(fp_counts_dict, total_frames, duration_seconds)` — or `None` if quit, or `({}, 0, 0)` if skipped.

In [ ]:
def run_fp_test(cap, seg_model, world_model) -> tuple | None:

    window    = "Benchmark"
    state     = IDLE
    # Count false-positive frames per class (initialised to 0 for all classes)
    fp_counts: dict[str, int] = {c: 0 for c in ALL_DANGER_CLASSES}
    total_frames = 0
    record_start = 0.0
    cached_world: list = []
    frame_count  = 0

    print(f"\n{'─'*55}")
    print("  FALSE POSITIVE TEST")
    print(f"  → Remove ALL dangerous objects from camera view")
    print(f"  → Record for {FP_TEST_SECONDS} seconds")
    print(f"  → Press SPACE when view is clear")
    print(f"{'─'*55}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        now = time.time()

        detected_set, seg_boxes, world_boxes = detect_classes_in_frame(
            frame, seg_model, world_model, frame_count, cached_world
        )

        annotated = frame.copy()

        # Highlight any danger-class detection as a false positive (they shouldn't be there)
        for cn, box in seg_boxes + world_boxes:
            if cn in ALL_DANGER_CLASSES:
                draw_detection_highlight(annotated, box, f"FP: {cn}", True)

        if state == IDLE:
            lines = [
                ("FALSE POSITIVE TEST — Remove all objects, then press SPACE",
                 (255, 200, 50)),
                ("S = skip  |  Q = quit", (160, 160, 160)),
            ]
            draw_status_bar(annotated, lines)
            draw_progress_bar(annotated, 0.0, (180, 180, 0))

        elif state == RECORDING:
            elapsed   = now - record_start
            remaining = max(0, FP_TEST_SECONDS - elapsed)
            total_frames += 1

            # Count each danger class that appears this frame as a false positive
            for cn in detected_set:
                if cn in fp_counts:
                    fp_counts[cn] += 1

            any_fp = bool(detected_set & set(ALL_DANGER_CLASSES))
            lines = [
                (f"RECORDING FP TEST  |  {remaining:.0f}s remaining", (255, 200, 50)),
                (f"Frames: {total_frames}  |  "
                 f"{'WARNING: FALSE POSITIVE DETECTED' if any_fp else 'Clean'}",
                 (80, 80, 255) if any_fp else (80, 220, 80)),
            ]
            draw_status_bar(annotated, lines)
            draw_progress_bar(annotated, elapsed / FP_TEST_SECONDS, (0, 0, 200))

            if elapsed >= FP_TEST_SECONDS:
                return fp_counts, total_frames, elapsed   # test complete

        cv2.imshow(window, annotated)

        key = cv2.waitKey(1) & 0xFF
        if key in [ord("q"), ord("Q"), 27]:
            return None
        elif key == ord(" ") and state == IDLE:
            state        = RECORDING
            record_start = time.time()
            total_frames = 0
            print("  ► Recording FP test…")
        elif key in [ord("s"), ord("S")]:
            print("  FP test skipped.")
            return {}, 0, 0

    return None

---
## Step 10 — Report Generation

**What:** Reads `benchmark_results.json` and produces two outputs:
1. A summary table printed to the terminal
2. A full Markdown report saved to `benchmark_report.md`

**Report structure:**

```
# Benchmark Report

## Detection Rate by Class and Distance
| Class | Source | Near DR | Mid DR | Far DR | Avg DR | Status |
|---|---|---|---|---|---|---|
| knife      | COCO seg  | **95%** ✅ | **88%** ✅ | 72% 🟡 | **85%** ✅ | ✅ Pass |
| lighter    | YOLOWorld | 61% 🔴    | 48% 🔴    | 22% 🔴 | **44%** 🔴 | 🔴 Needs training |
...

## False Positive Rate
| Class | FP Frames | FP / min |
...

## Summary
3 classes need custom training: lighter, pill bottle, button battery
```

**Colour coding in the report:**
- **Bold + ✅** — DR ≥ 85% (excellent)
- 🟡 — DR 70–84% (acceptable, passes threshold)
- **Bold + 🔴** — DR < 70% (fails, needs training)

**The Summary section is the most important output** — it gives you the exact list of classes to prioritise for Phase 2 data collection.

In [ ]:
def generate_report(results: dict) -> str:
    lines = []
    lines.append("# Benchmark Report — Child Safety Monitor")
    lines.append(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    lines.append(f"\nRecord duration per test: {RECORD_SECONDS}s  "
                 f"| Pass threshold: >= 70% detection rate\n")

    lines.append("## Detection Rate by Class and Distance\n")
    lines.append("| Class | Source | Near DR | Mid DR | Far DR | Avg DR | Status |")
    lines.append("|---|---|---|---|---|---|---|")

    needs_training = []

    for cls in ALL_DANGER_CLASSES:
        source = "COCO seg" if cls in COCO_CLASSES else "YOLOWorld"
        drs = {}
        for dist in DISTANCES:
            entry = results.get(cls, {}).get(dist)
            drs[dist] = entry["detection_rate"] if entry else None

        values = [v for v in drs.values() if v is not None]
        avg    = sum(values) / len(values) if values else None

        def fmt(v):
            if v is None:
                return "—"
            pct = v * 100
            if pct >= 85:  return f"**{pct:.0f}%** ✅"
            elif pct >= 70: return f"{pct:.0f}% 🟡"
            else:           return f"**{pct:.0f}%** 🔴"

        status = ("✅ Pass"            if avg is not None and avg >= 0.70
                  else ("🔴 Needs training" if avg is not None
                        else "⬜ Not tested"))

        if avg is not None and avg < 0.70:
            needs_training.append(cls)

        lines.append(f"| {cls} | {source} | {fmt(drs['near'])} | "
                     f"{fmt(drs['mid'])} | {fmt(drs['far'])} | {fmt(avg)} | {status} |")

    # False positive section
    fp_data = results.get("_false_positive_test")
    if fp_data:
        lines.append("\n## False Positive Rate\n")
        lines.append(f"Test duration: {fp_data.get('duration_seconds', '?')}s  "
                     f"| Total frames: {fp_data.get('total_frames', '?')}\n")
        lines.append("| Class | FP Frames | FP / min |")
        lines.append("|---|---|---|")
        fp_per_class = fp_data.get("fp_per_class", {})
        fp_per_min   = fp_data.get("fp_rate_per_min", {})
        for cls in ALL_DANGER_CLASSES:
            count = fp_per_class.get(cls, 0)
            rate  = fp_per_min.get(cls, 0.0)
            flag  = " ⚠️" if rate > 2.0 else ""   # flag classes with >2 FP/min
            lines.append(f"| {cls} | {count} | {rate:.1f}{flag} |")

    # Summary — the most actionable part of the report
    lines.append("\n## Summary\n")
    if needs_training:
        lines.append(f"**{len(needs_training)} class(es) need custom training (avg DR < 70%):**\n")
        for cls in needs_training:
            lines.append(f"- `{cls}`")
    else:
        lines.append("All tested classes meet the 70% detection rate threshold. ✅")

    lines.append("\n\n---")
    lines.append("*Re-run `python benchmark.py` to fill in untested slots.*")
    return "\n".join(lines)


def print_console_summary(results: dict) -> None:
    """Quick terminal table — shown after the benchmark finishes."""
    print("\n" + "═" * 60)
    print("  BENCHMARK SUMMARY")
    print("═" * 60)
    print(f"  {'Class':<25} {'Near':>7} {'Mid':>7} {'Far':>7} {'Avg':>7}")
    print("  " + "─" * 56)

    needs_training = []
    for cls in ALL_DANGER_CLASSES:
        drs = {}
        for dist in DISTANCES:
            entry = results.get(cls, {}).get(dist)
            drs[dist] = entry["detection_rate"] if entry else None

        values = [v for v in drs.values() if v is not None]
        avg    = sum(values) / len(values) if values else None

        def fmt(v):
            return f"{v*100:.0f}%" if v is not None else " —  "

        flag = ""
        if avg is not None and avg < 0.70:
            flag = " ← NEEDS TRAINING"
            needs_training.append(cls)

        print(f"  {cls:<25} {fmt(drs['near']):>7} {fmt(drs['mid']):>7}"
              f" {fmt(drs['far']):>7} {fmt(avg):>7}{flag}")

    print("═" * 60)
    if needs_training:
        print(f"\n  Classes below 70%: {', '.join(needs_training)}")
    print()

---
## Step 11 — Main Entry Point

**What:** Parses command-line arguments and orchestrates the full benchmark run.

**Three modes:**

| Command | Mode | What it does |
|---|---|---|
| `python benchmark.py` | Full benchmark | Runs all untested class × distance slots, then FP test |
| `python benchmark.py --report` | Report only | Reads saved results and generates report — no camera needed |
| `python benchmark.py --fp` | FP test only | Just the 60-second false positive test |

**Resume logic:**  
Before starting, the script calls `is_done()` on every slot in the test matrix to build `remaining` — the list of tests not yet completed. It prints how many are left and estimates how long they'll take. Already-completed slots are automatically skipped, so re-running the script never re-tests anything.

> ⚠️ This cell cannot run in Jupyter (uses `cv2.imshow` + `input()`). Run `python benchmark.py` from the terminal.

In [ ]:
# ⚠️ Cannot run in Jupyter — for reference only.

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--report", action="store_true",
                        help="Print report from saved results without running benchmark")
    parser.add_argument("--fp", action="store_true",
                        help="Run false-positive test only")
    args = parser.parse_args()

    results = load_results()   # load existing progress from disk

    # ── Report-only mode ─────────────────────────────────────────────────────
    if args.report:
        if not results:
            print("No results found. Run the benchmark first.")
            return
        print_console_summary(results)
        report_md = generate_report(results)
        REPORT_FILE.write_text(report_md)
        print(f"Full report saved to: {REPORT_FILE}")
        return

    # ── Load models ───────────────────────────────────────────────────────────
    print("\nLoading models…")
    seg_model   = YOLO("yolov8x-seg.pt")
    world_model = YOLOWorld("yolov8x-worldv2.pt")
    world_model.set_classes(CUSTOM_CLASSES)
    print("Models loaded.\n")

    # ── Camera ────────────────────────────────────────────────────────────────
    backend = cv2.CAP_AVFOUNDATION if platform.system() == "Darwin" else cv2.CAP_DSHOW
    cap = cv2.VideoCapture(CAMERA_INDEX, backend)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
    if not cap.isOpened():
        raise RuntimeError("Cannot open camera.")

    cv2.namedWindow("Benchmark", cv2.WINDOW_NORMAL | cv2.WINDOW_KEEPRATIO)

    # ── FP-only mode ──────────────────────────────────────────────────────────
    if args.fp:
        out = run_fp_test(cap, seg_model, world_model)
        if out and out[1] > 0:
            fp_counts, total_frames, duration = out
            record_fp_result(results, fp_counts, total_frames, duration)
            print("\nFP test saved.")
        cap.release()
        cv2.destroyAllWindows()
        return

    # ── Full benchmark ────────────────────────────────────────────────────────
    all_tests = [(cls, dist)
                 for cls in ALL_DANGER_CLASSES
                 for dist in DISTANCES]

    # Skip already-completed slots (resume support)
    remaining  = [(cls, dist) for cls, dist in all_tests
                  if not is_done(results, cls, dist)]
    done_count = len(all_tests) - len(remaining)

    print(f"Tests: {done_count}/{len(all_tests)} already complete.")
    print(f"Remaining: {len(remaining)} tests × {RECORD_SECONDS}s ≈ "
          f"{len(remaining)*RECORD_SECONDS//60} min\n")

    if remaining:
        print("Instructions:")
        print("  1. Place the specified object at the specified distance")
        print("  2. Press SPACE to begin recording (30 seconds)")
        print("  3. Keep the object visible; move it slightly during recording")
        print("  4. Press S to skip, Q to quit and save progress\n")
        input("Press ENTER to begin…")

    test_num       = done_count + 1
    quit_requested = False

    for cls, dist in remaining:
        result = run_single_test(
            cap, seg_model, world_model,
            cls, dist, test_num, len(all_tests)
        )

        if result is None:        # Q pressed — stop
            quit_requested = True
            break
        elif result == (-1, -1):  # S pressed — skip
            test_num += 1
            continue
        else:
            total_f, detected_f = result
            record_result(results, cls, dist, total_f, detected_f)
            dr     = detected_f / total_f if total_f else 0
            status = "PASS" if dr >= 0.70 else "FAIL"
            print(f"  → {cls} | {dist}: {dr*100:.1f}% [{status}]")
            test_num += 1

    # Offer FP test if detection tests are all done
    if not quit_requested and "_false_positive_test" not in results:
        print("\nAll detection tests done. Run false-positive test? (Y/N)")
        if input("> ").strip().lower() == "y":
            out = run_fp_test(cap, seg_model, world_model)
            if out and out[1] > 0:
                fp_counts, total_frames, duration = out
                record_fp_result(results, fp_counts, total_frames, duration)

    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

    # Print terminal summary and save markdown report
    print_console_summary(results)
    report_md = generate_report(results)
    REPORT_FILE.write_text(report_md)
    print(f"Full report saved to: {REPORT_FILE}\n")


if __name__ == "__main__":
    main()

---
## Summary — What the Benchmark Produces

After running, you will have:

| File | Content |
|---|---|
| `benchmark_results.json` | Raw data — detection rate per class per distance, FP counts |
| `benchmark_report.md` | Formatted table with pass/fail status per class |

### How to interpret the report for Phase 2 planning

| Result | Action |
|---|---|
| Avg DR ≥ 85% | Class is working well. No training needed. |
| Avg DR 70–84% | Acceptable. Can skip training if resources are limited. |
| Avg DR < 70% | **Prioritise for Phase 2 data collection.** |
| Far DR much lower than Near DR | Collect more far-distance training images specifically. |
| FP rate > 2/min | This class causes too many false alarms. Raise its confidence threshold or add it to Phase 2 training. |

### Expected results (before any training)

Based on known YOLOWorld zero-shot limitations:

| Class | Expected | Reason |
|---|---|---|
| knife, scissors | ✅ PASS | COCO-trained, many training examples |
| bottle, fork, oven | ✅ PASS | COCO-trained |
| lighter | 🔴 FAIL | Visually small, easily confused with other objects |
| pill bottle | 🔴 FAIL | Looks like a regular bottle — hard to distinguish without training |
| button battery | 🔴 FAIL | Tiny object, rarely appears in zero-shot training data |
| power cord | 🟡 borderline | Large enough to detect, but shape varies a lot |
| plastic bag | 🟡 borderline | Transparent/translucent — challenging for any model |